<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_02/sankey2_policysource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 2
### Policy source (skjult) → Target Group → Target → Land Use

#Import libaries

In [31]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors as pc


#Adgang til Airtable

In [32]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [33]:
# henter data fra Airtable via API og gemmer det i en pandas.DataFrame
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [34]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel2 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets
tabel3 = fetch_airtable_data("tblTRyuT48bBN24QG")   # Slut-land uses


# Fjern rækker med manglende værdier i Target Group-tabellen
tabel1 = tabel1.dropna(subset=['Target Group', 'Targets'])

# Fjern rækker med manglende værdier i Targets-tabellen
tabel2 = tabel2.dropna(subset=['Land uses'])

In [35]:
# ----- Forbered og normaliser tabeller (sikre navne + eksplosion af linked records) -----

# t0 = Policy Source
t0 = tabel0.copy()
# håndter variationer i kolonnenavn
if 'Policy source' in t0.columns:
    name_col = 'Policy source'
elif 'Policy Source' in t0.columns:
    name_col = 'Policy Source'
elif 'Name' in t0.columns:
    name_col = 'Name'
else:
    name_col = None

if name_col:
    t0 = t0.rename(columns={name_col: 'policy_source_name'})
# id -> policy_source_id
if 'id' in t0.columns:
    t0 = t0.rename(columns={'id': 'policy_source_id'})
# sørg for string-type for IDs
if 'policy_source_id' in t0.columns:
    t0['policy_source_id'] = t0['policy_source_id'].astype(str)


# t1 = Target Group
t1 = tabel1.copy()
# drop rækker uden nødvendige felter (brug de originale Airtable-navne du har)
if 'Target Group' in t1.columns and 'Targets' in t1.columns:
    t1 = t1.dropna(subset=['Target Group', 'Targets'])
# eksploder Targets (linked record IDs)
if 'Targets' in t1.columns:
    t1 = t1.explode('Targets').rename(columns={'Targets': 'target_id'})
# håndter evt. link til Policy Source i Target Group (meget vigtigt hvis vi vil filtrere)
if 'Policy source' in t1.columns:
    t1 = t1.explode('Policy source').rename(columns={'Policy source': 'policy_source_id'})
elif 'Policy Source' in t1.columns:
    t1 = t1.explode('Policy Source').rename(columns={'Policy Source': 'policy_source_id'})
# id for target-group
if 'id' in t1.columns:
    t1 = t1.rename(columns={'id': 'target_group_id'})
# konverter typer
if 'target_id' in t1.columns:
    t1['target_id'] = t1['target_id'].astype(str)
if 'policy_source_id' in t1.columns:
    t1['policy_source_id'] = t1['policy_source_id'].astype(str)


# t2 = Targets
t2 = tabel2.copy()
# drop rækker uden Land uses hvis nødvendigt
if 'Land uses' in t2.columns:
    t2 = t2.dropna(subset=['Land uses'])
    t2 = t2.explode('Land uses').rename(columns={'Land uses': 'land_use_id'})
# rename id -> target_id
if 'id' in t2.columns:
    t2 = t2.rename(columns={'id': 'target_id'})
# konverter typer
if 'target_id' in t2.columns:
    t2['target_id'] = t2['target_id'].astype(str)
if 'land_use_id' in t2.columns:
    t2['land_use_id'] = t2['land_use_id'].astype(str)


# t3 = Land Uses
t3 = tabel3.copy()
# find name-kolonne og rename til land_use_name hvis nødvendig
if 'Name' in t3.columns:
    t3 = t3.rename(columns={'Name': 'land_use_name'})
# rename id -> land_use_id
if 'id' in t3.columns:
    t3 = t3.rename(columns={'id': 'land_use_id'})
# konverter type
if 'land_use_id' in t3.columns:
    t3['land_use_id'] = t3['land_use_id'].astype(str)


# ----- Registrer i DuckDB med de navne du brugte tidligere -----
con.register('tabel0', t0)
con.register('tabel1', t1)
con.register('tabel1_exp', t2)
con.register('tabel2_exp', t3)


In [36]:
# Opret forbindelse
con = duckdb.connect()

# Registrer pandas DataFrames som tabeller i DuckDB
con.register('tabel0', t0)  # Policy Source
con.register('tabel1', t1)  # Target Group
con.register('tabel1_exp', t2)  # Targets
con.register('tabel2_exp', t3)  # Land Uses


In [37]:
SELECT
  t1."Target Group" AS target_group_name,
  t2."Name" AS target_name,
  t3.land_use_name,
  t0.policy_source_name
FROM tabel1 AS t1
JOIN tabel2_exp AS t2 ON t1.target_id = t2.target_id
JOIN tabel0 AS t0 ON t2.policy_source_id = t0.policy_source_id
JOIN tabel1_exp AS t3 ON t2.land_use_id = t3.land_use_id
WHERE t0.policy_source_name IN (
    'Aftale om et grønt Danmark (grøn trepart)',
    'Mere, bedre og større natur i Danmark',
    'Vandområdeplaner 2021-2027',
    'European Green Deal',
    'Common Agricultural Policy - Strategic plan 2023-2027',
    'EU biodiversity strategy 2030',
    'Technical Summary: Land Use and Climate Change',
    'Convention on Biological Diversity',
    'Common approach to integrating biodiversity and nature-based solutions for sustainable development into the United Nations policy and programme planning and delivery'
)


IndentationError: unexpected indent (ipython-input-3177534091.py, line 2)

In [ ]:
def forkort_label(label, max_len=40):
    if isinstance(label, str) and len(label) > max_len:
        return label[:max_len] + '…'
    return label

# Anvend på kolonner i join-resultat (fx result)
result['target_group_name_short'] = result['target_group_name'].apply(forkort_label)
result['land_use_name_short'] = result['land_use_name'].apply(forkort_label)

In [ ]:
# 1. Alle labels
all_labels = pd.concat([
    result['target_group_name_short'],
    result['target_name'],
    result['land_use_name_short']
]).unique().tolist()

# 2. Map label til indeks
label_to_index = {label: i for i, label in enumerate(all_labels)}

# 3. Kilde (source) og mål (target) for links
source = result['target_group_name_short'].map(label_to_index)
target = result['target_name'].map(label_to_index)
value = [1] * len(result)  # Vægt 1 pr række

# 4. Andet led links
source2 = result['target_name'].map(label_to_index)
target2 = result['land_use_name_short'].map(label_to_index)
value2 = [1] * len(result)

# 5. Saml alle links
source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.concat([pd.Series(value), pd.Series(value2)], ignore_index=True)

In [ ]:
# 1. Opret entydige node-id'er (tekniske ID'er – ikke labels)
result['node_TG'] = 'TG_' + result['target_group_name']
result['node_T'] = 'T_' + result['target_name']
result['node_LU'] = 'LU_' + result['land_use_name']

# 2. Saml alle tekniske node-ID'er i rækkefølge og uden gentagelser
all_node_ids = pd.concat([result['node_TG'], result['node_T'], result['node_LU']]).drop_duplicates().tolist()
node_to_index = {node_id: i for i, node_id in enumerate(all_node_ids)}

# 3. Brug tekniske ID'er til at mappe source/target
source = result['node_TG'].map(node_to_index)
target = result['node_T'].map(node_to_index)
source2 = target
target2 = result['node_LU'].map(node_to_index)

# 4. Saml source/target/value
value = [1] * len(result)
value2 = [1] * len(result)

source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.Series(value + value2)

# 5. Vis labels – pæne, forkortede versioner
def preprocess_label(label, max_len=40, wrap_len=60):
    if not isinstance(label, str):
        return label
    if len(label) > max_len:
        label = label[:max_len] + '…'
    return '\n'.join([label[i:i+wrap_len] for i in range(0, len(label), wrap_len)])

label_lookup = {
    **dict(zip(result['node_TG'], result['target_group_name'].apply(preprocess_label))),
    **dict(zip(result['node_T'], result['target_name'].apply(preprocess_label))),
    **dict(zip(result['node_LU'], result['land_use_name'].apply(preprocess_label)))
}
# Garanteret én label per node_id – i korrekt rækkefølge
all_labels = [label_lookup[node_id] for node_id in all_node_ids]

# 6. Farver
import plotly.colors as pc
node_colors = pc.qualitative.Plotly
node_colors_list = [node_colors[i % len(node_colors)] for i in range(len(all_labels))]

def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

link_colors = [lighten(node_colors_list[src], factor=0.8) for src in source_all]

# 7. Tegn Sankey-diagrammet
import plotly.graph_objects as go

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=80,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=all_labels,
        color=node_colors_list
    ),
    link=dict(
        source=source_all,
        target=target_all,
        value=value_all,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Target Group → Target → Land Use",
    font_size=12,
    height=6000
)

fig.show()


#Download


In [ ]:
fig.update_layout(title_text="Target Group → Target → Land Use", font_size=12, height=2000)
fig.write_html("sankey_diagram.html")

from google.colab import files
files.download("sankey_diagram.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
display(t0.head())
display(t1.head())
display(t2.head())
display(t3.head())

,policy_source_name,Source document,Source pdf,Publisher / author,Year,Territorial reference point,policy_source_id,Finished (step 1),Targets (policy targets),Checked (step 2),Checked (step 2) copy
0,The Impact of Disasters on Agriculture and Foo...,https://www.fao.org/publications/home/fao-flag...,"[{'id': 'attsWeD5BuwuWkuad', 'url': 'https://v...",Food and Agriculture Organization of the Unite...,2023,[Global],rec097wcwr7wH7Snq,NaN,NaN,NaN,NaN
1,Aftale om grøn omstilling af Dansk Landbrug,https://fm.dk/media/25302/aftale-om-groen-omst...,"[{'id': 'attupvM6YiCyZfrpg', 'url': 'https://v...",Regeringen,2021,[National],rec0UlvMlDgnViaLV,Finished,NaN,NaN,NaN
2,Climate Change 2023 - Synthesis Report,https://www.ipcc.ch/report/ar6/syr/downloads/r...,"[{'id': 'attDqMladH2i4q4HU', 'url': 'https://v...",Intergovernmental Panel on Climate Change,2023,[Global],rec1JMbQXty7j4wIq,NaN,NaN,NaN,NaN
3,Aftale om et grønt Danmark (grøn trepart),https://mim.dk/kampagner/groen-trepart,"[{'id': 'atto6IyXXispvRRbs', 'url': 'https://v...",Aftaleparterne i den grønne trepart,2024,[National],rec1ox4RpbP3H2cnG,NaN,"[recXapBmC0gLZzFW3, recFAwdYP1HIcS4XT, recm15m...",NaN,NaN
4,Global Assessment Report on Biodiversity and E...,https://www.ipbes.net/,"[{'id': 'attqMWASFv8O0jgNH', 'url': 'https://v...",Intergovernmental Science-Policy Platform on B...,2019,[Global],rec3Gwqr7J0lAy4Vy,NaN,NaN,NaN,NaN


,target_group_name,target_id,TARGET COUNT,target_group_id
0,Klimasikring,rechJaTmknKsHsYWP,19,rec2F1hkw7WEobaaO
0,Klimasikring,rec6HAufIoCtDZX2r,19,rec2F1hkw7WEobaaO
0,Klimasikring,recdAAHuvn4r571EB,19,rec2F1hkw7WEobaaO
0,Klimasikring,recPqQcrAgZpBb0Xm,19,rec2F1hkw7WEobaaO
0,Klimasikring,recwMRwBazGKlv49E,19,rec2F1hkw7WEobaaO


,target_name,Policy Source,Quotes (text excerpts with references),Description of target,Target Group,terest,Created,target_id,land_use_id,Target time frame,Functions,Land conditions
1,Sikring af følsomme arter og havbundshabitater...,[recO0BjJ3duCK3DUA],"""The application of an ecosystem-based managem...",Sikring af følsomme arter og havbundshabitater...,"[recQLsdhedkibpIKg, recSipJAiWSy8WqOL, recvEa8...",[Continental],2025-06-16T13:35:49.000Z,rec0D0BXryh6fVUbX,recNo1cCg9qZZHoRx,NaN,NaN,NaN
3,Beskyttelse af primære og gamle skovøkosysteme...,[recO0BjJ3duCK3DUA],"""As part of this focus on strict protection, i...",Primære og gamle skove er de rigeste skovøkosy...,[recxpWjra7wdZ6pu1],[Continental],2025-06-15T20:23:38.000Z,rec0JfZPYjBdr4YTr,recO482xvMhXKByv2,NaN,NaN,NaN
3,Beskyttelse af primære og gamle skovøkosysteme...,[recO0BjJ3duCK3DUA],"""As part of this focus on strict protection, i...",Primære og gamle skove er de rigeste skovøkosy...,[recxpWjra7wdZ6pu1],[Continental],2025-06-15T20:23:38.000Z,rec0JfZPYjBdr4YTr,recyVSA7AFt2aEv8K,NaN,NaN,NaN
3,Beskyttelse af primære og gamle skovøkosysteme...,[recO0BjJ3duCK3DUA],"""As part of this focus on strict protection, i...",Primære og gamle skove er de rigeste skovøkosy...,[recxpWjra7wdZ6pu1],[Continental],2025-06-15T20:23:38.000Z,rec0JfZPYjBdr4YTr,reclm66pMVI6K3w9v,NaN,NaN,NaN
3,Beskyttelse af primære og gamle skovøkosysteme...,[recO0BjJ3duCK3DUA],"""As part of this focus on strict protection, i...",Primære og gamle skove er de rigeste skovøkosy...,[recxpWjra7wdZ6pu1],[Continental],2025-06-15T20:23:38.000Z,rec0JfZPYjBdr4YTr,recXktaAG6Hd6elop,NaN,NaN,NaN


,land_use_name,Targets,Created,land_use_id,Land conditions (from Targets),X_Land conditions,Description of conditions needed for land to be suitable (from X_Land conditions)
0,Nye klimaresiliente afgrøder,[recDBl0og8nwVmACU],2025-06-22T18:14:58.000Z,rec0QqWf9Ueriq7Y7,NaN,NaN,NaN
1,Haver og parker,"[rec2IIEoRzoDUSppM, recw5eCczGf16YfTu, recnvJe...",2024-11-10T11:45:08.000Z,rec1PD6GZgfLPRDDs,NaN,NaN,NaN
2,Udendørs og helårsgræsning,"[rec4jaYz9OH7GoNQx, recvs06dRH2cLe4pI, recRc36...",2024-11-08T19:43:22.000Z,rec2BD715QVjrWSds,NaN,NaN,NaN
3,Braklægning,"[reccUNzKYSD5rHXaI, recPTNg97J0dQl8Sf, recItPZ...",2024-10-17T17:59:39.000Z,rec2bu0UfLTgbawLU,[med lav retention landbrugsområder i nærheden...,NaN,NaN
4,Blandingszoner,[recXuUD4EgCZgttzE],2024-11-10T14:46:07.000Z,rec4KvsG1TBOG42nI,NaN,NaN,NaN
